# Innerbloom: Qwen3.5-0.8B before and after weight editing

Scores the untrained Qwen3.5-0.8B on the 402 validation entries, then the version edited by the frontier model (see `ml/weight_editing/BRIEF.md`). Separate from notebook 02 because Qwen3.5 needs the newest `transformers`.

- Runtime > Change runtime type > **A100 GPU** (an L4 or T4 also works for 0.8B, more slowly and for fewer units).
- Run one cell at a time. **Disconnect and delete the runtime when finished.**

## 1. GPU, Drive, code and libraries (free, about 3 minutes)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/innerbloom/eval'
import os; os.makedirs(OUT, exist_ok=True)
!rm -rf /content/innerbloom && git clone -q --depth 1 -b feature/innerbloom-ai-v2 https://github.com/GhanshyamPaunikar/innerbloom /content/innerbloom
%cd /content/innerbloom
!pip install -q "transformers @ git+https://github.com/huggingface/transformers.git@main" accelerate requests

## 2. Untrained Qwen3.5-0.8B on validation (roughly 10-20 minutes, about 1-3 units)
The 'before' number. Ends with a summary.

In [ ]:
!python -m ml.eval_hf --model Qwen/Qwen3.5-0.8B --split validation --name qwen3.5-0.8b-untrained --out $OUT

## 3. Weight editing (done by the frontier model, not this notebook)
Give it `ml/weight_editing/BRIEF.md` and a sandbox like this one. It may only use the **train** split. Save its edited model to Drive as `MyDrive/innerbloom/eval/qwen3.5-0.8b-edited` and its `EDIT_LOG.md` next to it.

## 4. Edited model on validation (roughly 10-20 minutes, about 1-3 units)

In [ ]:
!python -m ml.eval_hf --model $OUT/qwen3.5-0.8b-edited --split validation --name qwen3.5-0.8b-edited --out $OUT

## 5. Compare everything
Adds the 7B (Ollama, from the repo) and the trained 1.5B (from notebook 02, if it has been run). Copy the table and send it back. Then **Runtime > Disconnect and delete runtime**.

In [ ]:
import os
paths = ['reports/runs/baseline-qwen2.5-7b-validation-v2/report.json',
         '/content/drive/MyDrive/innerbloom/phase3/qwen2.5-1.5b/comparison.json',
         f'{OUT}/qwen3.5-0.8b-untrained-validation/report.json',
         f'{OUT}/qwen3.5-0.8b-edited-validation/report.json']
!python -m ml.compare {' '.join(p for p in paths if os.path.exists(p))}